# PACER Grid Analysis — Amazon Clothing

Reusable post-hoc analysis for driver-emitted JSON payloads.

Compatible with:

| Grid | Driver | Output |
|---|---|---|
| P1+P2 | `scripts/run_p1_p2_nrdmc_grid.py` | `results/p1_p2_lambda_tau_grid_clothing.json` |
| P3 PTV | `scripts/run_p3_ptv_grid.py` | `results/p3_ptv_grid_clothing.json` |

Set `GRID_JSON` in cell 1 to point at whichever payload you want to
analyse. All charts are matplotlib (no seaborn) so the notebook runs
in a barebones Python env.


## 1. Load grid output

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# ---- CONFIG ----------------------------------------------------------------
# Point this at the driver's --out_json (relative to MMHCL_DAMPS_Project/).
GRID_JSON = "results/p3_ptv_grid_clothing.json"

# Control cell tag (used to compute effect-size vs baseline).
# For P3: "ctrl_k2"; for P1+P2: whichever cell you consider baseline
# (e.g. "l0p05_t0p30").
CONTROL_TAG = "ctrl_k2"

# Optional: pull additional WandB metrics.
WANDB_PROJECT = "damps-mmhcl-clothing"
WANDB_ENTITY = "baitapck51cc-uet"
WANDB_ENABLED = False  # flip to True if `pip install wandb` + logged in.
# ----------------------------------------------------------------------------

_here = Path(os.getcwd())
_candidates = [
    _here / GRID_JSON,
    _here.parent / GRID_JSON,
    _here / "MMHCL_DAMPS_Project" / GRID_JSON,
]
_json_path = next((p for p in _candidates if p.is_file()), None)
if _json_path is None:
    raise FileNotFoundError(
        f"Could not locate {GRID_JSON}. Tried: "
        + ", ".join(str(p) for p in _candidates)
    )

payload: dict[str, Any] = json.loads(_json_path.read_text(encoding="utf-8"))
print(f"Loaded {_json_path}")
print(f"  seeds: {payload.get('seeds')}")
print(f"  epoch cap: {payload.get('epoch')}")
print(f"  n configs: {len(payload.get('configs', []))}")
print(f"  n per_seed rows: {len(payload.get('per_seed', []))}")


## 2. Ranked leaderboard (test set)

In [ ]:
# Turn the driver's ``ranked`` array into a display DataFrame.
def _flat_stat(row: dict, key: str, stat: str) -> float:
    v = row.get(key, {})
    if isinstance(v, dict):
        return float(v.get(stat, float("nan")))
    return float("nan")

_ranked = payload.get("ranked", []) or []
_rows = []
for r in _ranked:
    _rows.append({
        "tag": r["tag"],
        "n_ok": int(r.get("n_ok", 0)),
        "R@20_mean": _flat_stat(r, "best_test_recall20", "mean"),
        "R@20_std":  _flat_stat(r, "best_test_recall20", "std"),
        "N@20_mean": _flat_stat(r, "best_test_ndcg20",   "mean"),
        "N@20_std":  _flat_stat(r, "best_test_ndcg20",   "std"),
        "Head_mean": _flat_stat(r, "test_head", "mean"),
        "Mid_mean":  _flat_stat(r, "test_mid",  "mean"),
        "Tail_mean": _flat_stat(r, "test_tail", "mean"),
        "K":         r.get("n_prototypes"),
        "lambda_ptv": r.get("lambda_ptv"),
    })
df_rank = pd.DataFrame(_rows)
df_rank_display = df_rank.copy()
for c in df_rank_display.columns:
    if c.endswith("_mean") or c.endswith("_std"):
        df_rank_display[c] = df_rank_display[c].map(
            lambda x: f"{x:.5f}" if not math.isnan(x) else "nan"
        )
df_rank_display


## 3. Effect size vs control

In [ ]:
# Delta R@20 and NDCG@20 vs the control tag.
if CONTROL_TAG not in df_rank["tag"].values:
    print(f"[WARN] control tag {CONTROL_TAG!r} not in ranked table; "
          "skipping effect-size analysis.")
    df_effect = pd.DataFrame()
else:
    ctrl = df_rank[df_rank.tag == CONTROL_TAG].iloc[0]
    _eff = []
    for _, r in df_rank.iterrows():
        if r["tag"] == CONTROL_TAG:
            continue
        dR = r["R@20_mean"] - ctrl["R@20_mean"]
        dN = r["N@20_mean"] - ctrl["N@20_mean"]
        pctR = 100.0 * dR / ctrl["R@20_mean"] if ctrl["R@20_mean"] > 0 else float("nan")
        pctN = 100.0 * dN / ctrl["N@20_mean"] if ctrl["N@20_mean"] > 0 else float("nan")
        _eff.append({
            "tag": r["tag"],
            "dR@20":  dR,
            "dR@20_pct": pctR,
            "dN@20":  dN,
            "dN@20_pct": pctN,
            "dHead":  r["Head_mean"] - ctrl["Head_mean"],
            "dMid":   r["Mid_mean"]  - ctrl["Mid_mean"],
            "dTail":  r["Tail_mean"] - ctrl["Tail_mean"],
        })
    df_effect = pd.DataFrame(_eff)
    for c in df_effect.columns:
        if c == "tag":
            continue
        if c.endswith("_pct"):
            df_effect[c] = df_effect[c].map(
                lambda x: f"{x:+.2f}%" if not math.isnan(x) else "nan"
            )
        else:
            df_effect[c] = df_effect[c].map(
                lambda x: f"{x:+.5f}" if not math.isnan(x) else "nan"
            )
df_effect


## 4. Paired-seed matrix

In [ ]:
# Long-form per_seed table, then pivot to (tag x seed) for R@20.
_ps = payload.get("per_seed", []) or []
df_ps = pd.DataFrame(_ps)
if df_ps.empty:
    print("[WARN] per_seed empty -- driver may have been --dry_run.")
else:
    print(f"per_seed rows: {len(df_ps)}   OK rows: {(df_ps.exit == 0).sum()}")
    # R@20 pivot
    _piv = df_ps.pivot_table(
        index="tag", columns="seed",
        values="best_test_recall20", aggfunc="first",
    )
    _piv = _piv.reindex(df_rank["tag"].tolist())  # preserve rank order
    _piv["mean"] = _piv.mean(axis=1)
    _piv["std"]  = _piv.iloc[:, :-1].std(axis=1, ddof=1)
    display_piv = _piv.copy()
    for c in display_piv.columns:
        display_piv[c] = display_piv[c].map(
            lambda x: f"{x:.5f}" if pd.notna(x) else "nan"
        )
    print("R@20 by (tag x seed):")
    display(display_piv)


## 5. Head / Mid / Tail decomposition

In [ ]:
_ter = df_rank[["tag", "Head_mean", "Mid_mean", "Tail_mean"]].copy()
if CONTROL_TAG in _ter.tag.values:
    ctrl_row = _ter[_ter.tag == CONTROL_TAG].iloc[0]
    _ter["dHead"] = _ter["Head_mean"] - ctrl_row["Head_mean"]
    _ter["dMid"]  = _ter["Mid_mean"]  - ctrl_row["Mid_mean"]
    _ter["dTail"] = _ter["Tail_mean"] - ctrl_row["Tail_mean"]
for c in _ter.columns[1:]:
    _ter[c] = _ter[c].map(
        lambda x: f"{x:+.5f}" if isinstance(x, float) and c.startswith("d")
        else (f"{x:.5f}" if isinstance(x, float) else x)
    )
_ter


## 6. Paired significance (Wilcoxon vs control)

With only 2-5 seeds per cell the parametric paired *t*-test is unreliable;
we use the non-parametric Wilcoxon signed-rank test as a floor.
For 2 seeds Wilcoxon is under-powered but still directional; treat p-values
as advisory, not confirmatory.

In [ ]:
from scipy import stats  # type: ignore

if CONTROL_TAG not in df_ps.tag.unique():
    print(f"[WARN] control {CONTROL_TAG!r} missing in per_seed -- skipping.")
else:
    ctrl_rows = df_ps[df_ps.tag == CONTROL_TAG].sort_values("seed")
    sig_rows = []
    for tag, grp in df_ps.groupby("tag"):
        if tag == CONTROL_TAG:
            continue
        grp = grp.sort_values("seed")
        seeds_common = sorted(
            set(grp.seed).intersection(set(ctrl_rows.seed))
        )
        if len(seeds_common) < 2:
            sig_rows.append({"tag": tag, "n": len(seeds_common),
                             "median_dR20": float("nan"),
                             "wilcoxon_p": float("nan")})
            continue
        a = grp.set_index("seed").loc[seeds_common, "best_test_recall20"].values
        b = ctrl_rows.set_index("seed").loc[seeds_common, "best_test_recall20"].values
        diffs = a - b
        try:
            _, p = stats.wilcoxon(a, b, zero_method="wilcox")
        except (ValueError, RuntimeError):
            p = float("nan")
        sig_rows.append({
            "tag": tag,
            "n": len(seeds_common),
            "median_dR20": float(np.median(diffs)),
            "wilcoxon_p": p,
        })
    df_sig = pd.DataFrame(sig_rows)
    df_sig["median_dR20"] = df_sig["median_dR20"].map(
        lambda x: f"{x:+.5f}" if pd.notna(x) else "nan"
    )
    df_sig["wilcoxon_p"] = df_sig["wilcoxon_p"].map(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )
    display(df_sig)


## 7. R@20 bar chart with 95% CI

In [ ]:
import matplotlib.pyplot as plt

# Nexus palette
PRIMARY  = "#20808D"  # chart teal
ACCENT   = "#A84B2F"  # rust (control)
GRID_C   = "#D4D1CA"
TEXT_C   = "#28251D"

tags = df_rank["tag"].tolist()
means = df_rank["R@20_mean"].values
stds  = df_rank["R@20_std"].values
n_seeds = len(payload.get("seeds", []))
# 95% CI ~ 1.96 * std / sqrt(n)
ci = 1.96 * stds / max(math.sqrt(max(n_seeds, 1)), 1.0)
colors = [ACCENT if t == CONTROL_TAG else PRIMARY for t in tags]

fig, ax = plt.subplots(figsize=(9, 4.5), dpi=110)
bars = ax.bar(tags, means, yerr=ci, color=colors,
              edgecolor="none", capsize=4)
ax.set_ylabel("Test Recall@20", color=TEXT_C)
ax.set_title(f"Test Recall@20 by cell — {_json_path.name}", color=TEXT_C)
ax.tick_params(colors=TEXT_C)
ax.grid(True, axis="y", color=GRID_C, linewidth=0.8, alpha=0.7)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(GRID_C)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 8. Head / Mid / Tail grouped bars

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=110)
x = np.arange(len(tags))
w = 0.27
h_vals = df_rank["Head_mean"].values
m_vals = df_rank["Mid_mean"].values
t_vals = df_rank["Tail_mean"].values

ax.bar(x - w, h_vals, w, color="#1B474D", label="Head")
ax.bar(x,     m_vals, w, color="#20808D", label="Mid")
ax.bar(x + w, t_vals, w, color="#BCE2E7", label="Tail")

ax.set_xticks(x)
ax.set_xticklabels(tags, rotation=30, ha="right", color=TEXT_C)
ax.set_ylabel("Recall@20 (per tercile)", color=TEXT_C)
ax.set_title("Popularity tercile breakdown", color=TEXT_C)
ax.legend(frameon=False, loc="upper right")
ax.grid(True, axis="y", color=GRID_C, linewidth=0.8, alpha=0.7)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(GRID_C)
plt.tight_layout()
plt.show()


## 9. Wall-clock per cell (sanity check for speedup)

In [ ]:
if df_ps.empty or "wall_min" not in df_ps.columns:
    print("[WARN] no wall_min column -- driver too old, or dry-run.")
else:
    _wall = df_ps.groupby("tag")["wall_min"].agg(["mean", "std", "count"])
    _wall = _wall.reindex(df_rank["tag"].tolist())
    print(_wall.round(2))

    fig, ax = plt.subplots(figsize=(9, 3.5), dpi=110)
    ax.bar(_wall.index, _wall["mean"].values,
           yerr=_wall["std"].fillna(0).values,
           color=PRIMARY, edgecolor="none", capsize=4)
    ax.set_ylabel("wall-clock per seed (min)", color=TEXT_C)
    ax.set_title("Per-cell training wall-clock", color=TEXT_C)
    ax.grid(True, axis="y", color=GRID_C, linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 10. WandB audit — `compile_attached` filter (optional)

If WandB is available, pull the runs matching this grid's tags and check
whether `torch.compile` was actually attached (introduced in commit
`bca72df`). Cross-verifies the local JSON against the persistent WandB
record.

In [ ]:
if not WANDB_ENABLED:
    print("Set WANDB_ENABLED = True in cell 1 to run this section.")
else:
    import wandb
    api = wandb.Api()
    runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}",
                    per_page=200, order="-created_at")
    tag_set = set(df_rank["tag"].tolist())
    matched = []
    for r in runs:
        name = getattr(r, "name", "") or ""
        # Match on any cell tag appearing in the run name.
        hits = [t for t in tag_set if t in name]
        if not hits:
            continue
        cfg = r.config
        matched.append({
            "run": name[:60],
            "cell": hits[0],
            "compile_attached": cfg.get("compile_attached"),
            "cuda_graph_enabled": cfg.get("cuda_graph_enabled"),
            "state": r.state,
            "created": str(r.created_at)[:19],
        })
    if not matched:
        print("[WARN] no matching WandB runs found.")
    else:
        df_wb = pd.DataFrame(matched).sort_values("created", ascending=False)
        display(df_wb.head(30))
        # Sanity summary: how many runs actually got compile attached?
        _ok = df_wb["compile_attached"].fillna(False).sum()
        print(f"\ncompile_attached=True in {int(_ok)}/{len(df_wb)} runs")


## 11. Verdict template

Fill in after reviewing the tables and charts above.

**Winner:** `<tag>` — R@20 = `<mean>` ± `<std>` (n=`<seeds>`)

**Effect vs control (`"""{CONTROL_TAG}"""`):** dR@20 = `<...>` (`<+X.XX%>`)

**Where it wins:** ☐ Head / ☐ Mid / ☐ Tail

**Wilcoxon p (advisory, n small):** `<p>` — treat as `<significant | inconclusive>`

**Wall-clock cost:** `<mean_min>` min/seed × `<seeds>` seeds × `<cells>` cells = `<total_h>` h

**Compile audit (from cell 10):** `<X>/<Y>` runs had `compile_attached=True`.

**Next step:**  
- If winner beats control by >= 1% and Wilcoxon `p < 0.1`: promote to Section `<X>` in the paper, run 5-seed replication.  
- If winner within noise: mark cell as dominated, drop from paper.  
- If Head/Tail trade-off is severe (>2% loss on any tercile): revisit fusion weights.
